# Assignment 2: Colab Fine-Tuning Notebook

This notebook is self-contained. Upload only this file to Google Colab, switch the runtime to `GPU`, and run the cells in order.

It will:
- create the custom e-commerce support dataset
- fine-tune `TinyLlama/TinyLlama-1.1B-Chat-v1.0` with LoRA
- evaluate the base model and fine-tuned model
- save artifacts to Google Drive if enabled


## 1. Mount Drive And Create Workspace

Set `USE_DRIVE = True` if you want the dataset and model artifacts to persist in Drive.


In [ ]:
from pathlib import Path
import os

USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/assignment_2_ecommerce_finetuning')
else:
    PROJECT_DIR = Path('/content/assignment_2_ecommerce_finetuning')

for subdir in ['scripts', 'data/ecommerce_support', 'artifacts']:
    (PROJECT_DIR / subdir).mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_DIR)
print('Working directory:', PROJECT_DIR)


## 2. Install Dependencies


In [ ]:
!pip -q install torch transformers datasets accelerate peft sentencepiece scikit-learn tqdm pandas


## 3. Generate The Custom Dataset

This creates a custom synthetic dataset with `720` examples and train/validation/test splits.


In [ ]:
from __future__ import annotations

import csv
import json
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

SEED = 42
STORE_NAME = 'ShopSphere'
EXAMPLES_PER_CATEGORY = 40
TRAIN_PER_CATEGORY = 32
VAL_PER_CATEGORY = 4
TEST_PER_CATEGORY = 4
OUTPUT_DIR = PROJECT_DIR / 'data' / 'ecommerce_support'

INTRO_PHRASES = ['Hi,', 'Hello,', 'Hey support,', 'Good morning,', 'Good evening,', '']
ENDINGS = ['Please help.', 'What should I do?', 'Please advise.', 'Can you check this for me?', 'I need support with this.', 'Kindly guide me.']
CUSTOMER_TYPES = ['I am a first-time customer.', 'I order from your app regularly.', 'This is my first order on your platform.', 'I use your website often.', '']
PRODUCTS = ['wireless earbuds', 'gaming mouse', 'Bluetooth speaker', 'running shoes', 'smartwatch', 'phone case', 'laptop backpack', 'electric kettle', 'yoga mat', 'USB-C charger', 'desk lamp', 'mechanical keyboard', 'air fryer', 'skin care kit', 'power bank']
SIZES = ['XS', 'S', 'M', 'L', 'XL', 'UK 7', 'UK 8', 'UK 9', '128GB', '256GB']
COLORS = ['black', 'blue', 'green', 'pink', 'white', 'gray', 'navy']
PAYMENT_METHODS = ['UPI', 'credit card', 'debit card', 'net banking', 'wallet', 'COD']
COUPONS = ['SAVE10', 'WELCOME15', 'FESTIVE20', 'APPONLY5', 'STYLE25', 'NEWUSER50']
ISSUES = ['the app keeps showing an error', 'the tracking page has not updated', 'I cannot see any useful status on the order page', 'the website is giving me confusing messages']
TIME_PHRASES = ['yesterday', 'two days ago', 'three days ago', 'last night', 'this morning']
DELIVERY_WINDOWS = ['the promised delivery date was yesterday', 'it was supposed to arrive today but there is still no delivery partner update', 'the estimated delivery window has already passed', 'the order is now later than the original delivery date']
RETURN_REASONS = ['I changed my mind', 'the product does not match my expectation', 'I no longer need it', 'I ordered the wrong variant by mistake']
LOGIN_ISSUES = ['I am not receiving the OTP', 'the password reset link is not working', 'the app signs me out every time I log in', 'I keep getting an invalid credentials message']

CATEGORY_SPECS = [
    {
        'name': 'order_tracking',
        'prompts': [
            'I placed an order for a {product} and I want to track where it is. My order ID is {order_id}.',
            'Can you tell me the current status of order {order_id} for my {product}?',
            'I ordered a {product} {time_phrase} and I want to know how to track it.',
            'I need a tracking update for order {order_id} because {issue}.'
        ],
        'responses': [
            "Please open My Orders and use the tracking link to see the courier's latest scan. If the tracking page does not update for more than 48 hours, contact support with your order ID so the shipment can be escalated.",
            'You can track the shipment from the order details page in your account. If there is no movement for 48 hours or the tracking link is blank, share the order ID with support for a delivery investigation.',
            'The fastest way to check the package status is from the tracking section under your order details. If the status stays unchanged for more than 48 hours, contact support so the courier can be followed up.'
        ]
    },
    {
        'name': 'delayed_delivery',
        'prompts': [
            'My order for a {product} is delayed and {delivery_window}.',
            'Order {order_id} has not arrived yet even though {delivery_window}.',
            'I was expecting my {product} by now, but the parcel still has not arrived.',
            'My shipment looks stuck in transit and {delivery_window}.'
        ],
        'responses': [
            'I am sorry your order is delayed. Please check the latest tracking update first, and if the parcel is still not delivered 48 hours after the promised date, contact support with your order ID so the delay can be escalated.',
            'Please review the courier tracking page from My Orders for the latest scan. If the order remains undelivered for 48 hours beyond the expected date, contact support so the shipment can be investigated.',
            'A short delay can happen in transit, but the order should be escalated if it is still pending 48 hours after the promised delivery date. Please share the order ID with support if that happens.'
        ]
    },
    {
        'name': 'cancel_before_shipment',
        'prompts': [
            'I want to cancel my order for the {product} before it ships.',
            'Can I cancel order {order_id} right now? It still looks like it has not been shipped.',
            'I placed a {product} order by mistake and I need to cancel it before dispatch.',
            'My order is still in processing status. How do I cancel it?'
        ],
        'responses': [
            'If the order status is still Confirmed or Processing, you can cancel it from My Orders. Once the cancellation is accepted, the refund starts automatically to the original payment method.',
            'You can cancel the order yourself from the order details page as long as it has not been shipped. After the cancellation is confirmed, the refund is initiated automatically.',
            'Cancellation is available only before shipment. Please open My Orders and tap cancel if the order is still in Processing or Confirmed status, and the refund will begin automatically after that.'
        ]
    },
    {
        'name': 'cancel_after_shipment',
        'prompts': [
            'My order for the {product} has already shipped, but I want to cancel it now.',
            'Can you cancel order {order_id} even though the tracking page says it is shipped?',
            'I no longer want the {product}, but it is already out for delivery.',
            'The order is already in transit and I need to cancel it. What are my options?'
        ],
        'responses': [
            'Once an order has been shipped, it cannot be canceled in the system. You can refuse the delivery or accept it and start a return request within 7 days of delivery if the item is eligible.',
            'Shipped orders cannot be canceled anymore. If you do not want the package, you can refuse delivery or place a return request within 7 days after delivery for eligible items.',
            'Cancellation is not available after shipment. Please refuse the parcel at delivery or submit a return request after delivery within the 7-day return window if the item is eligible.'
        ]
    },
    {
        'name': 'return_request',
        'prompts': [
            'I received my {product}, but {return_reason} and I want to return it.',
            'How can I return order {order_id}? The item was delivered recently.',
            'I want to place a return request for a {product} that was delivered {time_phrase}.',
            'Please tell me the return process for my {product}.'
        ],
        'responses': [
            'You can request a return from My Orders within 7 days of delivery if the item is eligible and in its original condition. Please keep the product, tags, and packaging ready for pickup.',
            'Open the delivered order in My Orders and select Return to start the process. Returns are allowed within 7 days of delivery for eligible items that are unused and in original condition.',
            'Please raise the return request from your order details page within 7 days of delivery. Make sure the item is unused and packed with all original tags and accessories for a smooth pickup.'
        ]
    },
    {
        'name': 'refund_status',
        'prompts': [
            'I returned order {order_id} and I want to know when my refund will come.',
            'My return pickup is complete, but I still do not see the refund.',
            'I am checking the refund status for a returned item. The payment method was {payment_method}.',
            'The return was approved a few days ago. How long does the refund take?'
        ],
        'responses': [
            'Refunds are processed after the returned item passes inspection. For prepaid orders, the amount usually reaches the original payment method within 5 to 7 business days, while COD refunds are sent to the selected bank account or UPI ID within 3 to 5 business days after approval.',
            'Once the return is verified, the refund is started automatically. Prepaid refunds usually take 5 to 7 business days to reflect, and COD refunds are completed to the registered bank or UPI details within 3 to 5 business days after approval.',
            'Please allow the return quality check to finish first. After approval, prepaid refunds normally take 5 to 7 business days, and COD refunds take about 3 to 5 business days to the bank account or UPI details provided.'
        ]
    },
    {
        'name': 'exchange_request',
        'prompts': [
            'I ordered a {product} in size {size}, but I need size {size_2}. Can I exchange it?',
            'I want to exchange my {color} {product} for {color_2}.',
            'My {product} was delivered, but I need a different size or color. How do I exchange it?',
            'Can you help me exchange order {order_id} for another variant of the same item?'
        ],
        'responses': [
            'You can request an exchange from My Orders within 7 days of delivery if exchange is available for that product. Exchanges are completed only when the requested size or color is in stock; otherwise you can return the item for a refund.',
            'Please open the delivered order and check whether Exchange is available for that item. If the new variant is in stock, the exchange can be placed within 7 days of delivery, and if not, you can return the item instead.',
            'Exchange requests are allowed within 7 days of delivery for eligible products and only when the requested variant is available. If the size or color is out of stock, please place a return request for a refund.'
        ]
    },
    {
        'name': 'wrong_item_received',
        'prompts': [
            'I ordered a {product}, but I received a {product_2} instead.',
            'Order {order_id} contains the wrong item. What should I do?',
            'The package delivered today has a different product than what I purchased.',
            'I received the wrong variant in my order and I need this fixed.'
        ],
        'responses': [
            'I am sorry you received the wrong item. Please report it within 48 hours of delivery from My Orders and upload clear photos of the product, packaging, and shipping label so a replacement or refund can be arranged.',
            'Please raise a Wrong Item issue within 48 hours of delivery and attach photos of the received product, outer package, and label. Once verified, support can arrange a replacement or refund.',
            'If the delivered item does not match the order, report the issue within 48 hours from the order details page. Clear photos of the item and package are needed so the case can be verified for replacement or refund.'
        ]
    },
    {
        'name': 'damaged_item',
        'prompts': [
            'My {product} arrived damaged today.',
            'Order {order_id} was delivered, but the item is broken.',
            'I opened my package and the {product} is damaged. How do I report this?',
            'The product I received has visible damage and I need a replacement or refund.'
        ],
        'responses': [
            'I am sorry the item arrived damaged. Please report the damage within 48 hours of delivery and upload clear photos or an unboxing video from My Orders so the case can be verified for a replacement or refund.',
            'Please open the order details page and submit a Damaged Item request within 48 hours of delivery. Clear product photos, package photos, and any unboxing video will help support process the replacement or refund faster.',
            'Damage issues must be reported within 48 hours of delivery from the order page. Please attach photos of the product and packaging, and include an unboxing video if available, so the request can be reviewed for refund or replacement.'
        ]
    },
    {
        'name': 'missing_item',
        'prompts': [
            'My package was delivered, but the {product} is missing from the box.',
            'Order {order_id} arrived with one item missing.',
            'I received the parcel today, but one product is not inside the package.',
            'The box looked incomplete and my item is missing. What should I do next?'
        ],
        'responses': [
            'Please report the missing item within 48 hours of delivery from My Orders. Upload photos of the outer package, shipping label, and everything that was inside the box so the shortage can be investigated.',
            'If an item is missing from a delivered order, raise the issue within 48 hours and attach clear photos of the package, label, and received contents. Support will review the case and arrange a refund or replacement if it is verified.',
            'A missing item complaint should be submitted within 48 hours of delivery from the order details page. Please share package photos and the shipping label so the logistics team can investigate the shortage.'
        ]
    },
    {
        'name': 'payment_failed',
        'prompts': [
            'My payment failed while using {payment_method}, and the order did not go through.',
            'I was trying to place an order, but the checkout failed at the payment step.',
            'The app showed a payment failure message when I tried to pay by {payment_method}.',
            'I cannot complete my purchase because the payment page keeps failing.'
        ],
        'responses': [
            'Please try placing the order again after checking your payment details and internet connection, or switch to another payment method. If no money was deducted and no order was created, the failed attempt does not need any further action.',
            'A failed payment usually means the order was not placed. Please retry the checkout once, and if the issue continues, use another payment option or contact your bank or payment provider.',
            'If the payment failed and no amount was deducted, you can safely retry the order or use a different payment method. If the error keeps repeating, contact support with a screenshot of the checkout error.'
        ]
    },
    {
        'name': 'payment_deducted_no_order',
        'prompts': [
            'My payment was deducted through {payment_method}, but no order was created.',
            'I paid for checkout, but the app crashed and I cannot see any order. The transaction ID is {transaction_id}.',
            'Money has been debited from my account, but I never got an order confirmation.',
            'The payment went through, but the order page is empty. What should I do now?'
        ],
        'responses': [
            'If the payment was deducted but the order was not created, the amount is usually reversed automatically within 3 to 5 business days. If the refund is not received after that, contact support with the payment reference or bank transaction ID.',
            'Please wait for 3 to 5 business days because most failed checkout deductions are auto-reversed by the bank or payment gateway. If the amount is still not credited back after that, share the transaction ID with support for manual review.',
            'A deduction without order confirmation is normally refunded automatically within 3 to 5 business days. If there is no reversal after that period, contact support with the transaction details so the payment team can investigate.'
        ]
    },
    {
        'name': 'address_change',
        'prompts': [
            'I entered the wrong shipping address for order {order_id}. How can I change it?',
            'I want to update my delivery address before the order ships.',
            'The pin code on my order is wrong and I need to correct the address.',
            'Can I edit the address for my current order? It still looks like it has not been dispatched.'
        ],
        'responses': [
            'You can change the delivery address only before the order is shipped. Please check My Orders for the edit option, and if the order is already packed or shipped, the address can no longer be changed.',
            'Address updates are allowed only until shipment. Open the order details page and edit the address if that option is visible; once the order is shipped, the delivery address cannot be modified.',
            'Please try updating the address from My Orders while the order is still in Confirmed or Processing status. If the shipment has already started, the address cannot be changed anymore.'
        ]
    },
    {
        'name': 'coupon_not_working',
        'prompts': [
            'The coupon code {coupon} is not working on my cart.',
            'I am trying to use {coupon}, but checkout says the coupon is invalid.',
            'My discount code is failing even though I typed it correctly.',
            'Why is the coupon not applying to order {order_id}?'
        ],
        'responses': [
            'Please check whether the coupon is still valid, meets the minimum cart value, and is eligible for the products in your cart. Only one coupon can be used per order, and some brands or categories may be excluded from discounts.',
            'Coupon errors usually happen when the code is expired, the cart value is below the minimum requirement, or the items are excluded from offers. Please also make sure only one coupon is being applied to the order.',
            'Please verify the coupon expiry, minimum spend, and product eligibility first. Discount codes cannot be combined, and some sellers or categories are excluded from promotional offers.'
        ]
    },
    {
        'name': 'warranty_claim',
        'prompts': [
            'My {product} has stopped working and I want to claim warranty.',
            'How do I start a warranty claim for order {order_id}?',
            'The product was delivered earlier, but now it has developed a fault. What is the warranty process?',
            'I need support for a warranty issue with my {product}.'
        ],
        'responses': [
            'Please contact support with your order ID, a short description of the fault, and photos or video if the issue is visible. Warranty claims are handled based on the seller or brand policy, and the team will guide you on repair, replacement, or service center steps.',
            'To begin a warranty claim, share the order ID, product issue, and any supporting photos or video with support. The next step depends on the seller or brand warranty terms, and you will be guided on repair or replacement options.',
            'Please keep the invoice and order ID ready and contact support with a clear description of the issue. If the defect is visible, include photos or video so the warranty process can be reviewed under the brand or seller policy.'
        ]
    },
    {
        'name': 'out_of_stock_query',
        'prompts': [
            'The {product} I want to buy is out of stock. When will it be available again?',
            'Can you tell me if the {product} will be restocked soon?',
            'I am waiting for a {product}, but it still shows out of stock.',
            'Do you reserve stock for customers, or should I wait for the item to come back?'
        ],
        'responses': [
            'Restock timing depends on the seller and current inventory, so support cannot promise a fixed date. Please use the Notify Me option on the product page so you receive an alert if the item becomes available again.',
            'Out-of-stock items are restocked based on seller inventory updates, so there is no guaranteed date. Please enable stock alerts on the product page and check back later for availability.',
            'We cannot confirm a restock date in advance because availability depends on the seller. The best option is to turn on Notify Me for that product so you are informed when the item returns.'
        ]
    },
    {
        'name': 'account_login_issue',
        'prompts': [
            'I cannot log in to my account because {login_issue}.',
            'I am locked out of my shopping account and need help getting back in.',
            'The app is not letting me sign in even though I am using the correct details.',
            'How do I recover access to my account if login is failing?'
        ],
        'responses': [
            'Please use Forgot Password or the OTP login option first and make sure you are using the registered email address or phone number. If you still cannot log in, clear the app cache, try again, and contact support if the issue continues.',
            'Start with password reset or OTP login using your registered contact details. If the OTP does not arrive or the app keeps failing, clear cache or try the website version before contacting support for account assistance.',
            'Please confirm that you are using the correct registered email or phone number, then try the reset password or OTP option. If login still fails after clearing cache or trying another browser, contact support for further verification.'
        ]
    },
    {
        'name': 'invoice_request',
        'prompts': [
            'I need the invoice for my {product} order.',
            'How can I download the bill for order {order_id}?',
            'I need a GST invoice or order invoice for reimbursement.',
            'The item was delivered, but I cannot find the invoice in my account.'
        ],
        'responses': [
            'You can download the invoice from My Orders once the item is shipped or delivered. Open the order details page and look for the Invoice option, and if it is still missing after 24 hours, contact support with the order ID.',
            'Please check the order details page in My Orders for the invoice download option. If the invoice is not visible within 24 hours after shipment or delivery, contact support so it can be reviewed.',
            'Invoices are usually available from the order details page after shipment or delivery. If you still do not see the invoice after 24 hours, share the order ID with support for help.'
        ]
    }
]

def clean_text(text: str) -> str:
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.replace(' ,', ',').replace(' .', '.').replace(' ?', '?')
    return text

def order_id(rng: random.Random) -> str:
    return f'SS{rng.randint(100000, 999999)}'

def transaction_id(rng: random.Random) -> str:
    return f'TXN{rng.randint(10000000, 99999999)}'

def prompt_with_style(rng: random.Random, body: str) -> str:
    parts = [rng.choice(INTRO_PHRASES), body, rng.choice(CUSTOMER_TYPES), rng.choice(ENDINGS)]
    return clean_text(' '.join(part for part in parts if part))

def build_vars(rng: random.Random) -> dict:
    return {
        'product': rng.choice(PRODUCTS),
        'product_2': rng.choice(PRODUCTS),
        'order_id': order_id(rng),
        'transaction_id': transaction_id(rng),
        'time_phrase': rng.choice(TIME_PHRASES),
        'issue': rng.choice(ISSUES),
        'delivery_window': rng.choice(DELIVERY_WINDOWS),
        'return_reason': rng.choice(RETURN_REASONS),
        'payment_method': rng.choice(PAYMENT_METHODS),
        'size': rng.choice(SIZES),
        'size_2': rng.choice(SIZES),
        'color': rng.choice(COLORS),
        'color_2': rng.choice(COLORS),
        'coupon': rng.choice(COUPONS),
        'login_issue': rng.choice(LOGIN_ISSUES),
    }

rng = random.Random(SEED)
examples = []

for spec in CATEGORY_SPECS:
    category_rows = []
    seen_prompts = set()
    while len(category_rows) < EXAMPLES_PER_CATEGORY:
        values = build_vars(rng)
        prompt_body = rng.choice(spec['prompts']).format(**values)
        prompt = prompt_with_style(rng, prompt_body)
        if prompt.lower() in seen_prompts:
            continue
        seen_prompts.add(prompt.lower())
        category_rows.append({
            'category': spec['name'],
            'prompt': prompt,
            'response': rng.choice(spec['responses']),
        })
    examples.extend(category_rows)

grouped = defaultdict(list)
for row in examples:
    grouped[row['category']].append(row)

splits = {'train': [], 'validation': [], 'test': []}
example_id = 1
for category in sorted(grouped):
    rows = grouped[category][:]
    rng.shuffle(rows)
    split_map = {
        'train': rows[:TRAIN_PER_CATEGORY],
        'validation': rows[TRAIN_PER_CATEGORY:TRAIN_PER_CATEGORY + VAL_PER_CATEGORY],
        'test': rows[TRAIN_PER_CATEGORY + VAL_PER_CATEGORY:],
    }
    for split_name, split_rows in split_map.items():
        for row in split_rows:
            splits[split_name].append({
                'id': f'ecs_{example_id:04d}',
                'dataset_name': 'ecommerce_customer_support_assistant',
                'domain': 'e-commerce customer support',
                'store_name': STORE_NAME,
                'split': split_name,
                **row,
            })
            example_id += 1

splits['all'] = splits['train'] + splits['validation'] + splits['test']

def write_jsonl(path: Path, rows: list[dict]) -> None:
    with path.open('w', encoding='utf-8') as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=True) + '\n')

def write_csv(path: Path, rows: list[dict]) -> None:
    if not rows:
        return
    with path.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

for split_name in ['all', 'train', 'validation', 'test']:
    write_jsonl(OUTPUT_DIR / f'{split_name}.jsonl', splits[split_name])
    write_csv(OUTPUT_DIR / f'{split_name}.csv', splits[split_name])

stats = {
    'dataset_name': 'ecommerce_customer_support_assistant',
    'store_name': STORE_NAME,
    'total_examples': len(splits['all']),
    'split_counts': {name: len(splits[name]) for name in ['train', 'validation', 'test']},
    'category_counts': dict(Counter(row['category'] for row in splits['all']))
}
(OUTPUT_DIR / 'stats.json').write_text(json.dumps(stats, indent=2), encoding='utf-8')

print(json.dumps(stats, indent=2))
print('Dataset directory:', OUTPUT_DIR)


In [ ]:
import pandas as pd

train_df = pd.read_json(PROJECT_DIR / 'data' / 'ecommerce_support' / 'train.jsonl', lines=True)
display(train_df.head(5))
print('Train rows:', len(train_df))
print('Unique categories:', train_df['category'].nunique())


## 4. Fine-Tune With LoRA

This cell trains a small instruct model. If Colab memory is tight, reduce batch size or set `MAX_LENGTH = 384`.


In [ ]:
from __future__ import annotations

import inspect
import json
import math
import random

import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, DataCollatorForSeq2Seq, Trainer, TrainingArguments

MODEL_NAME = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
OUTPUT_DIR = PROJECT_DIR / 'artifacts' / 'tinyllama_ecommerce_lora'
TRAIN_FILE = PROJECT_DIR / 'data' / 'ecommerce_support' / 'train.jsonl'
VALIDATION_FILE = PROJECT_DIR / 'data' / 'ecommerce_support' / 'validation.jsonl'
SYSTEM_PROMPT = 'You are a helpful e-commerce customer support assistant for ShopSphere. Answer clearly, politely, and only according to the store policy shown in the training data. Give short practical next steps and do not invent unsupported policies.'
MAX_LENGTH = 384
SEED = 42

def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def build_prompt(user_prompt: str, system_prompt: str) -> str:
    return f'<|system|>\n{system_prompt}\n<|user|>\n{user_prompt.strip()}\n<|assistant|>\n'

def tokenize_example(example: dict, tokenizer, max_length: int, system_prompt: str) -> dict:
    prompt_text = build_prompt(example['prompt'], system_prompt)
    response_text = example['response'].strip() + tokenizer.eos_token
    prompt_tokens = tokenizer(prompt_text, add_special_tokens=False)
    response_tokens = tokenizer(response_text, add_special_tokens=False)
    input_ids = prompt_tokens['input_ids'] + response_tokens['input_ids']
    attention_mask = prompt_tokens['attention_mask'] + response_tokens['attention_mask']
    labels = [-100] * len(prompt_tokens['input_ids']) + response_tokens['input_ids']
    return {
        'input_ids': input_ids[:max_length],
        'attention_mask': attention_mask[:max_length],
        'labels': labels[:max_length],
    }

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    if isinstance(logits, tuple):
        logits = logits[0]
    predictions = torch.tensor(logits).argmax(dim=-1)
    labels = torch.tensor(labels)
    shifted_predictions = predictions[:, :-1].contiguous()
    shifted_labels = labels[:, 1:].contiguous()
    valid_mask = shifted_labels != -100
    if valid_mask.sum().item() == 0:
        return {'token_accuracy': 0.0}
    correct = ((shifted_predictions == shifted_labels) & valid_mask).sum().item()
    total = valid_mask.sum().item()
    return {'token_accuracy': correct / total}

set_seed(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

torch_dtype = torch.float16 if torch.cuda.is_available() else None
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch_dtype)
model.config.use_cache = False

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

dataset = load_dataset('json', data_files={'train': str(TRAIN_FILE), 'validation': str(VALIDATION_FILE)})
tokenized = dataset.map(
    lambda example: tokenize_example(example, tokenizer=tokenizer, max_length=MAX_LENGTH, system_prompt=SYSTEM_PROMPT),
    remove_columns=dataset['train'].column_names,
)

training_kwargs = {
    'output_dir': str(OUTPUT_DIR),
    'learning_rate': 2e-4,
    'num_train_epochs': 2,
    'per_device_train_batch_size': 2,
    'per_device_eval_batch_size': 2,
    'gradient_accumulation_steps': 8,
    'weight_decay': 0.01,
    'warmup_ratio': 0.1,
    'logging_steps': 10,
    'eval_steps': 25,
    'save_steps': 25,
    'save_total_limit': 2,
    'load_best_model_at_end': True,
    'metric_for_best_model': 'eval_loss',
    'greater_is_better': False,
    'report_to': 'none',
    'remove_unused_columns': False,
    'fp16': torch.cuda.is_available(),
    'seed': SEED,
}

training_args_signature = inspect.signature(TrainingArguments.__init__)
if 'evaluation_strategy' in training_args_signature.parameters:
    training_kwargs['evaluation_strategy'] = 'steps'
elif 'eval_strategy' in training_args_signature.parameters:
    training_kwargs['eval_strategy'] = 'steps'

training_args = TrainingArguments(**training_kwargs)
trainer_kwargs = {
    'model': model,
    'args': training_args,
    'train_dataset': tokenized['train'],
    'eval_dataset': tokenized['validation'],
    'data_collator': DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
    'compute_metrics': compute_metrics,
}
trainer_signature = inspect.signature(Trainer.__init__)
if 'tokenizer' in trainer_signature.parameters:
    trainer_kwargs['tokenizer'] = tokenizer
elif 'processing_class' in trainer_signature.parameters:
    trainer_kwargs['processing_class'] = tokenizer

trainer = Trainer(**trainer_kwargs)
train_result = trainer.train()
metrics = trainer.evaluate()
metrics['train_loss'] = train_result.training_loss
if 'eval_loss' in metrics:
    metrics['eval_perplexity'] = math.exp(metrics['eval_loss'])

trainer.save_model()
tokenizer.save_pretrained(OUTPUT_DIR)
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print(json.dumps(metrics, indent=2))
print('Saved adapter to:', OUTPUT_DIR)


## 5. Compare Base Model And Fine-Tuned Model


In [ ]:
import json
import re
from pathlib import Path

import pandas as pd
import torch
from peft import PeftModel
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

TEST_FILE = PROJECT_DIR / 'data' / 'ecommerce_support' / 'test.jsonl'
BASE_OUTPUT = PROJECT_DIR / 'artifacts' / 'base_eval.json'
TUNED_OUTPUT = PROJECT_DIR / 'artifacts' / 'finetuned_eval.json'

def normalize_text(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def exact_match(prediction: str, reference: str) -> float:
    return float(normalize_text(prediction) == normalize_text(reference))

def token_f1(prediction: str, reference: str) -> float:
    pred_tokens = normalize_text(prediction).split()
    ref_tokens = normalize_text(reference).split()
    if not pred_tokens or not ref_tokens:
        return 0.0
    pred_counts = {}
    ref_counts = {}
    for token in pred_tokens:
        pred_counts[token] = pred_counts.get(token, 0) + 1
    for token in ref_tokens:
        ref_counts[token] = ref_counts.get(token, 0) + 1
    overlap = 0
    for token, count in pred_counts.items():
        overlap += min(count, ref_counts.get(token, 0))
    if overlap == 0:
        return 0.0
    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

def build_prompt(user_prompt: str, system_prompt: str) -> str:
    return f'<|system|>\n{system_prompt}\n<|user|>\n{user_prompt.strip()}\n<|assistant|>\n'

def load_rows(path: Path) -> list[dict]:
    rows = []
    with path.open('r', encoding='utf-8') as handle:
        for line in handle:
            if line.strip():
                rows.append(json.loads(line))
    return rows

def load_model(model_name: str, adapter_path: Path | None = None):
    tokenizer = AutoTokenizer.from_pretrained(str(adapter_path) if adapter_path else model_name, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    torch_dtype = torch.float16 if torch.cuda.is_available() else None
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch_dtype)
    if adapter_path:
        model = PeftModel.from_pretrained(model, str(adapter_path))
    model.eval()
    return tokenizer, model

def generate_answer(tokenizer, model, prompt: str, max_new_tokens: int = 120) -> str:
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

def evaluate_model(model_name: str, adapter_path: Path | None, output_path: Path) -> dict:
    rows = load_rows(TEST_FILE)
    tokenizer, model = load_model(model_name, adapter_path)
    exact_scores = []
    f1_scores = []
    predictions = []
    for row in tqdm(rows, desc=f'Evaluating {adapter_path.name if adapter_path else "base"}'):
        prompt = build_prompt(row['prompt'], SYSTEM_PROMPT)
        prediction = generate_answer(tokenizer, model, prompt)
        em = exact_match(prediction, row['response'])
        f1 = token_f1(prediction, row['response'])
        exact_scores.append(em)
        f1_scores.append(f1)
        predictions.append({
            'id': row['id'],
            'category': row['category'],
            'prompt': row['prompt'],
            'reference': row['response'],
            'prediction': prediction,
            'exact_match': em,
            'token_f1': f1,
        })
    summary = {
        'model_name': model_name,
        'adapter_path': str(adapter_path) if adapter_path else None,
        'examples_evaluated': len(predictions),
        'exact_match': sum(exact_scores) / len(exact_scores),
        'token_f1': sum(f1_scores) / len(f1_scores),
        'predictions': predictions,
    }
    output_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
    return summary

base_summary = evaluate_model(MODEL_NAME, adapter_path=None, output_path=BASE_OUTPUT)
tuned_summary = evaluate_model(MODEL_NAME, adapter_path=OUTPUT_DIR, output_path=TUNED_OUTPUT)

print('Base metrics:', {k: v for k, v in base_summary.items() if k != 'predictions'})
print('Fine-tuned metrics:', {k: v for k, v in tuned_summary.items() if k != 'predictions'})

comparison_rows = []
for base_row, tuned_row in zip(base_summary['predictions'][:10], tuned_summary['predictions'][:10]):
    comparison_rows.append({
        'category': tuned_row['category'],
        'prompt': tuned_row['prompt'],
        'reference': tuned_row['reference'],
        'base_prediction': base_row['prediction'],
        'fine_tuned_prediction': tuned_row['prediction'],
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)
